In [ ]:
import torch

device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(device)

SEED = 42
torch.manual_seed(SEED)
split_generator = torch.Generator().manual_seed(SEED)

In [ ]:
import os

from data.augment import AudioAugmenter, AugmentConfig
from data.load_data import (
    DistillDataset,
    TwitDataset,
    collate_fn,
    get_class_imbalance,
    make_augmenting_collate
)
from torch.utils.data import DataLoader

ds = TwitDataset()

train_ds, test_ds = torch.utils.data.random_split(
    ds,
    [0.9, 0.1],
    generator=split_generator,
)

train_source = DistillDataset(train_ds, f"distill/targets/perch.npz")

# Preserve a 50% clean path and soften each transform when augmentation is selected.
augment_config = AugmentConfig(
    sample_rate=16_000,
    apply_prob=0.50,
    time_shift_prob=0.35,
    background_mix_prob=0.35,
    background_snr_db=(5.0, 20.0),
    pink_noise_prob=0.25,
    pink_noise_snr_db=(10.0, 30.0),
    level_dbfs=(-45.0, -20.0),
)

train_collate = make_augmenting_collate(AudioAugmenter(augment_config))

In [ ]:
NUM_WORKERS = min(4, os.cpu_count() or 1)
loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    persistent_workers=NUM_WORKERS > 0,
    pin_memory=(device.type == "cuda"),
    batch_size=32
)
train_dataloader = DataLoader(
    train_source,
    shuffle=True,
    collate_fn=train_collate,
    **loader_kwargs,
)
test_dataloader = DataLoader(
    test_ds,
    shuffle=False,
    collate_fn=collate_fn,
    **loader_kwargs,
)

pos_weight = get_class_imbalance(train_ds).to(device)

In [ ]:
from models import DSCNNHead, GaborFilter, GaborNet, SpecAugment
from torch import nn

N_FILTERS = 40
SAMPLE_RATE = 32000     # must match the (resampled) audio fed to the model
KERNEL_SIZE = 401       # ~25 ms @ 16 kHz: long enough to resolve ~1 kHz carriers
STRIDE = 160


feat_extract = GaborFilter(n_filters=N_FILTERS, kernel_size=KERNEL_SIZE, sample_rate=SAMPLE_RATE, stride=STRIDE)
head = DSCNNHead(channels=(32, 64, 64), activation=nn.LeakyReLU)
model = GaborNet(feat_extract, head).to(device)
model.spec_augment = SpecAugment(
    freq_masks=1,
    time_masks=1,
    max_freq_fraction=0.15,
    max_time_fraction=0.10,
    apply_prob=0.50,
).to(device)

In [ ]:
from train import Trainer

trainer = Trainer(
    model,
    train_dataloader,
    test_dataloader,
    pos_weight,
    device,
    lr=1e-2,
    lr_scheduler_kwargs={"eta_min": 1e-4},
    seed=SEED,
    distill_alpha=0.5,
    distill_temp=2.0,
    warmup_epochs=5,
    run_name="dscnn/32k-sample-rate",
)

trainer.train(epochs=50)

In [ ]:
trainer.save_model()